# Progressive Cramming — interactive demo**Compress a span of text into a single learnable *memory embedding* of a frozen language model — and watch it reconstruct.**This notebook is a hands-on companion to the paper *Progressive Cramming: Reliable Token Compression and What It Reveals*. It runs on a free **Colab T4 GPU**.> ⚠️ **Requires a GPU.** In Colab: **Runtime ▸ Change runtime type ▸ T4 GPU**, then run the cells top to bottom.What you can do here:1. **Setup** — install + load `Llama-3.2-1B`.2. **What is cramming** — the one-picture idea.3. **Pre-compressed gallery** — reconstruct 5 domains from embeddings we cached on the Hub.4. **Total vs Progressive cramming** — side-by-side on one example.5. **Compress your own text** — live optimisation + PCA trajectory + reconstruction.6. **Advanced** — alignment / low-dim / horizon controls.

---## 1 · SetupInstall the package and the small UI dependency, then load the frozen model. The first install pulls `transformers`/`torch` and takes a couple of minutes.

In [ ]:
# Config — edit here if you fork the demo.MODEL_CHECKPOINT = "unsloth/Llama-3.2-1B"  # ungated Llama-3.2-1B mirror (no HF license gate)GALLERY_REPO_ID  = "LarryLovestein/progressive_cramming_demo_gallery"  # pre-computed embeddingsMAX_SEQ_LEN      = 64  # interactive cramming cap (keeps the demo snappy on a T4)REPO_URL         = "https://github.com/FusionBrainLab/progressive_cramming"SLIDES_URL       = "https://fusionbrainlab.github.io/progressive_cramming/slides/"# Models offered in section 5 ("compress your own text"). All are ungated and fit a# Colab T4 for cramming a short span (paper families: Pythia / SmolLM2 / Llama-3 / Gemma-3).# The gallery (sections 3-4) is tied to MODEL_CHECKPOINT and cannot use the others.INTERACTIVE_MODELS = {    "SmolLM2-135M":         "HuggingFaceTB/SmolLM2-135M",    "Pythia-160M":          "EleutherAI/pythia-160m",    "Gemma-3-270M":         "unsloth/gemma-3-270m",    "SmolLM2-360M":         "HuggingFaceTB/SmolLM2-360M",    "Pythia-410M":          "EleutherAI/pythia-410m",    "Gemma-3-1B":           "unsloth/gemma-3-1b-pt",    "Llama-3.2-1B (default)": "unsloth/Llama-3.2-1B",    "Pythia-1.4B":          "EleutherAI/pythia-1.4b",    "SmolLM2-1.7B":         "HuggingFaceTB/SmolLM2-1.7B",}

In [ ]:
%pip install -q "git+https://github.com/FusionBrainLab/progressive_cramming.git" ipywidgets

In [ ]:
import torchassert torch.cuda.is_available(), (    "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then re-run.")print("GPU:", torch.cuda.get_device_name(0))from progressive_cramming.demo import (    load_frozen_model, cram_text, progressive_cram_text, reconstruct_text,)# fp16 runs natively on a T4 (Turing has no hardware bf16); the compression# embedding itself is always optimised in float32 inside cram_text.model, tokenizer = load_frozen_model(MODEL_CHECKPOINT, dtype="float16")print("Loaded", MODEL_CHECKPOINT, "| hidden size:", model.config.hidden_size)

A couple of small display helpers used throughout (token-level coloured diff and an embedding loader).

In [ ]:
import htmlimport torchfrom IPython.display import HTML, displaydef emb_from_row(row):    """Dataset row -> compression embedding tensor [n_cram, hidden] (float32)."""    return torch.tensor(row["embedding"], dtype=torch.float32)def _strip_bos(ids, tok):    bos = tok.bos_token_id    ids = list(ids)    if bos is not None and ids and ids[0] == bos:        ids = ids[1:]    return idsdef render_token_diff(gt_ids, gen_ids, title="", tok=None):    """Colour the reconstruction token-by-token: green = matches the original at that position, red = differs."""    tok = tok or tokenizer    g = _strip_bos(gt_ids, tok)    h = list(gen_ids)    spans = []    for i in range(max(len(g), len(h))):        a = g[i] if i < len(g) else None        b = h[i] if i < len(h) else None        piece = tok.decode([b]) if b is not None else ""        disp = html.escape(piece).replace("\n", "⏎")        color = "#d6f5d6" if (a is not None and b is not None and a == b) else "#f8d0d0"        spans.append(f'<span style="background:{color};padding:1px 2px;border-radius:2px">{disp}</span>')    return (f'<div style="font-family:monospace;line-height:1.9;margin:6px 0">'            f'<b>{html.escape(title)}</b><br>{"".join(spans)}</div>')def reconstruct_and_show(row, label, m=None, tok=None):    """Greedy-decode from a row's embedding and render the coloured diff vs the original."""    m = m or model    tok = tok or tokenizer    gen = reconstruct_text(m, tok, emb_from_row(row), max_new_tokens=row["num_tokens"] + 4)    gen_ids = tok(gen, add_special_tokens=False)["input_ids"]    display(HTML(render_token_diff(row["input_ids"], gen_ids, label, tok=tok)))

---## 2 · What is cramming?**Cramming** freezes a pretrained language model and optimises a small set of input embeddings — by gradient descent — until the frozen model reconstructs the original tokens from them. The weights never change; all the information is squeezed into the optimised **memory embedding**.**Progressive cramming** grows the target span stage by stage, advancing only once the current span reconstructs *exactly* — which pins each text's **compression horizon** (the longest prefix one embedding can hold). The new embedding is *warm-started* from the previous stage:![Progressive cramming](https://raw.githubusercontent.com/FusionBrainLab/progressive_cramming/main/page/assets/fig_progressive.png)📄 [Paper & code](https://github.com/FusionBrainLab/progressive_cramming) · 🎬 [Video slides](https://fusionbrainlab.github.io/progressive_cramming/slides/) · 🗂 [Trajectories dataset](https://huggingface.co/datasets/mrsndmn/progressive_cramming_trajectories)

---## 3 · Pre-compressed galleryFive **paragraph-sized** spans from different domains (literature from PG19, code, news, poetry, science), each already crammed into **one** memory embedding (we cached them on the Hub so you don't wait for optimisation). Hit **▶ Reconstruct** to greedily decode each one back. Green = the token matches the original at that position.

In [ ]:
from datasets import load_datasetimport ipywidgets as widgetsgallery_ds = load_dataset(GALLERY_REPO_ID, split="train")gallery_rows = [r for r in gallery_ds if r["kind"] == "gallery"]def make_card(row):    out = widgets.Output()    btn = widgets.Button(description="▶ Reconstruct", button_style="primary")    header = widgets.HTML(        f"<b>{html.escape(row['domain'])}</b> &mdash; {html.escape(row['title'])}"        f"<br><span style='color:#666;font-family:monospace'>{html.escape(row['text'][:140])}</span>"    )    def on_click(_):        out.clear_output()        with out:            print("Reconstructing from 1 memory embedding...")            reconstruct_and_show(row, "Reconstruction (green = exact match)")            print(f"reconstruction={row['final_convergence']:.3f}  |  "                  f"info gain={row['information_gain_bits']:.0f} bits  |  "                  f"{row['num_tokens']} tokens → {row['n_cram']} embedding")    btn.on_click(on_click)    return widgets.VBox([header, btn, out],                        layout=widgets.Layout(border="1px solid #ddd", padding="10px", margin="6px 0"))display(widgets.VBox([make_card(r) for r in gallery_rows]))

---## 4 · Total vs Progressive crammingOne **longer passage** (~140 tokens — beyond the comfort zone of a single embedding on this small model), two procedures:- **Total cramming (TC)** — optimise one embedding to reconstruct the *whole* span at once. It reaches high teacher-forced accuracy, but a residual early-token error makes **greedy decoding drift** (the paper's brittleness finding, Table 1).- **Progressive cramming (PC)** — grow the span stage by stage and stop at the **compression horizon** (the longest prefix it reconstructs *exactly*). That prefix decodes back cleanly under greedy generation.Both are decoded greedily from their cached embeddings and coloured against the original — watch where TC turns red while PC stays green up to its horizon.

In [ ]:
tc_pc_rows = [r for r in gallery_ds if r["kind"] == "tc_pc"]tc_row = next(r for r in tc_pc_rows if r["method"] == "full_cramming")pc_row = next(r for r in tc_pc_rows if r["method"] == "progressive_cramming")out4 = widgets.Output()btn4 = widgets.Button(description="▶ Run side-by-side", button_style="primary")def run_side_by_side(_):    out4.clear_output()    with out4:        display(HTML(f"<b>Original ({html.escape(tc_row['domain'])}):</b> "                     f"<span style='font-family:monospace'>{html.escape(tc_row['text'])}</span>"))        reconstruct_and_show(tc_row, "Total cramming — whole span at once")        reconstruct_and_show(pc_row, "Progressive cramming — grown to the horizon")        display(HTML(            f"<small>TC steps-to-converge: <b>{tc_row['steps_taken']}</b> &middot; "            f"PC compression horizon: <b>{pc_row['horizon']}/{pc_row['num_tokens']}</b> tokens</small>"))btn4.on_click(run_side_by_side)display(widgets.VBox([btn4, out4]))

---## 5 · Compress your own textType anything, **pick a model**, choose a length cap, and hit **▶ Compress**. You'll watch the optimisation *live*: the loss/reconstruction curves on the left, and the embedding's **PCA trajectory** through optimisation on the right (black = init, red ★ = converged). Then we decode it back, colour the diff, report the metrics, and let you download the embedding.The model dropdown lets you compare cramming across the paper's families at sizes that fit a T4 (Pythia / SmolLM2 / Llama-3.2 / Gemma-3) — smaller models converge faster but hold fewer tokens. The first use of each new model downloads it (a minute or so); the gallery model stays loaded for sections 3–4.This runs the actual cramming loop (a transparent re-implementation built from the package's primitives) — capped at a short span so it stays interactive on a T4.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom sklearn.decomposition import PCAfrom IPython.display import clear_outputimport gctext_input  = widgets.Textarea(    value="Cramming compresses a span of text into a single learnable embedding of a frozen model.",    layout=widgets.Layout(width="100%", height="80px"))model_dd     = widgets.Dropdown(options=list(INTERACTIVE_MODELS), value="Llama-3.2-1B (default)", description="Model")len_slider   = widgets.IntSlider(value=32, min=8, max=MAX_SEQ_LEN, step=4, description="Max tokens")steps_slider = widgets.IntSlider(value=1500, min=200, max=3000, step=100, description="Max steps")run_btn      = widgets.Button(description="▶ Compress", button_style="primary")out5         = widgets.Output()last_result  = {}# Interactive-model cache. The gallery model (MODEL_CHECKPOINT) is kept loaded for# sections 3-4; at most one *other* model is held at a time to stay within T4 memory._interactive = {"ckpt": MODEL_CHECKPOINT, "model": model, "tokenizer": tokenizer}def get_interactive_model(ckpt):    if _interactive["ckpt"] == ckpt and _interactive["model"] is not None:        return _interactive["model"], _interactive["tokenizer"]    if ckpt == MODEL_CHECKPOINT:        chosen = (model, tokenizer)  # reuse the always-loaded gallery model    else:        # Free the previously loaded non-gallery model before loading a new one.        if _interactive["model"] is not None and _interactive["ckpt"] != MODEL_CHECKPOINT:            _interactive["model"] = None            _interactive["tokenizer"] = None            gc.collect(); torch.cuda.empty_cache()        print(f"Loading {ckpt} ...")        chosen = load_frozen_model(ckpt, dtype="float16")    _interactive.update({"ckpt": ckpt, "model": chosen[0], "tokenizer": chosen[1]})    return chosenclass LiveViz:    """Accumulates per-step metrics + embedding snapshots and redraws curves + PCA path live."""    def __init__(self, redraw_every=20):        self.steps, self.losses, self.convs = [], [], []        self.snaps, self.snap_steps = [], []        self.redraw_every, self._since = redraw_every, 0    def __call__(self, info):        self.steps.append(info["step"]); self.losses.append(info["loss"]); self.convs.append(info["convergence"])        if info["captured"]:            self.snaps.append(info["embedding"].reshape(-1).float().cpu().numpy())            self.snap_steps.append(info["step"])        self._since += 1        if self._since >= self.redraw_every:            self._since = 0            self.draw()    def draw(self):        clear_output(wait=True)        fig, ax = plt.subplots(1, 2, figsize=(11, 4))        ax[0].plot(self.steps, self.losses, color="tab:red")        ax[0].set_xlabel("step"); ax[0].set_ylabel("loss", color="tab:red"); ax[0].set_title("Optimisation")        axb = ax[0].twinx(); axb.plot(self.steps, self.convs, color="tab:green")        axb.set_ylabel("reconstruction", color="tab:green"); axb.set_ylim(-0.02, 1.02)        if len(self.snaps) >= 2:            P = PCA(n_components=2).fit_transform(np.stack(self.snaps))            ax[1].plot(P[:, 0], P[:, 1], color="gray", alpha=0.4, lw=1)            sc = ax[1].scatter(P[:, 0], P[:, 1], c=self.snap_steps, cmap="viridis", s=18)            ax[1].scatter([P[0, 0]], [P[0, 1]], color="black", marker="o", label="init")            ax[1].scatter([P[-1, 0]], [P[-1, 1]], color="red", marker="*", s=160, label="converged")            fig.colorbar(sc, ax=ax[1], label="step"); ax[1].legend()        ax[1].set_title("Embedding trajectory (PCA)")        plt.tight_layout(); plt.show(); plt.close(fig)def on_run(_):    out5.clear_output()    with out5:        ckpt = INTERACTIVE_MODELS[model_dd.value]        m, t = get_interactive_model(ckpt)        viz = LiveViz()        capture = max(5, steps_slider.value // 120)        res = cram_text(m, t, text_input.value,                        max_seq_len=len_slider.value, max_steps=steps_slider.value,                        capture_every=capture, on_step=viz)        viz.draw()        gen = reconstruct_text(m, t, res.embedding, max_new_tokens=res.num_tokens + 4)        gen_ids = t(gen, add_special_tokens=False)["input_ids"]        display(HTML(render_token_diff(res.input_ids, gen_ids, "Reconstruction (green = exact match)", tok=t)))        display(HTML(            f"<b>model:</b> {html.escape(model_dd.value)} &middot; "            f"<b>n_cram:</b> {res.num_mem_tokens} embedding &middot; "            f"<b>tokens:</b> {res.num_tokens} &middot; "            f"<b>reconstruction:</b> {res.final_convergence:.3f} &middot; "            f"<b>info gain:</b> {res.information_gain_bits:.0f} bits &middot; "            f"<b>steps:</b> {res.steps_taken} &middot; <b>time:</b> {res.elapsed_s:.1f}s"))        last_result["res"] = res        display(dl_btn)dl_btn = widgets.Button(description="⬇ Download embedding (.pt)")def on_dl(_):    res = last_result.get("res")    if res is None:        return    torch.save({"embedding": res.embedding, "input_ids": res.input_ids,                "text": res.text, "training_config": res.training_config()}, "my_embedding.pt")    try:        from google.colab import files        files.download("my_embedding.pt")    except Exception:        print("Saved to my_embedding.pt")dl_btn.on_click(on_dl)run_btn.on_click(on_run)display(widgets.VBox([text_input, model_dd, widgets.HBox([len_slider, steps_slider]), run_btn, out5]))

---## 6 · AdvancedThe paper's variants are selected with a few flags on the full repo. Pick options below to assemble the exact command. Run it in a clone of [the repository](https://github.com/FusionBrainLab/progressive_cramming) (a GPU job, heavier than this demo).- **Activation alignment (hybrid)** — also match the frozen model's hidden states, not just the output tokens.- **Low-dim projection (K)** — optimise the embedding inside a learned rank-K subspace instead of the full hidden size.

In [ ]:
method_dd  = widgets.Dropdown(options=["full", "progressive", "low-dim"], value="full", description="method")lowdim_k   = widgets.IntSlider(value=32, min=8, max=256, step=8, description="low-dim K")align_chk  = widgets.Checkbox(value=False, description="activation alignment (hybrid)")build_btn  = widgets.Button(description="Build command", button_style="primary")out6       = widgets.Output()def build_cmd(_):    out6.clear_output()    with out6:        flags = [            "python scripts/run_cramming.py",            f"--model_checkpoint {MODEL_CHECKPOINT}",            "--dataset_name LarryLovestein/pg19_1k --limit_dataset_items 4",            f"--max_sequence_length {MAX_SEQ_LEN}",            "--embedding_init_method random0.02 --learning_rate 0.1",        ]        if method_dd.value == "progressive":            flags.append("--progressive_train 1 --progressive_step 8 --max_optimization_steps_per_token 500")        if method_dd.value == "low-dim":            flags.append(f"--low_dim_train --low_dim_size {lowdim_k.value}")        if align_chk.value:            flags.append("--loss_type cosine --hybrid_alpha 1.0 --num_alignment_layers 8")        print(" \\\n    ".join(flags))build_btn.on_click(build_cmd)display(widgets.VBox([method_dd, lowdim_k, align_chk, build_btn, out6]))

---### Citation```bibtex@inproceedings{tarasov2026progressive,  title     = {Progressive Cramming: Reliable Token Compression and What It Reveals},  author    = {Tarasov, Dmitrii and Lashukov and Goncharova, Elena and Kuznetsov, Andrey},  year      = {2026},}```